# 🤝 Giorno 2 - Lab 2: Protocollo A2A (Agent2Agent) & Coordinamento Multi-Agente

Benvenuti al secondo laboratorio del **Giorno 2**!
Nella sessione precedente abbiamo visto come un agente comunica con i propri Tool tramite **MCP** (standard trasparente per dotare il singolo agente di mani e occhi).

In questo notebook esploreremo **A2A (Agent2Agent)**:
- Il protocollo open-source per far collaborare **agenti autonomi ed eterogenei** (spesso appartenenti a reparti, team o fornitori diversi).
- A differenza di MCP (dove ogni tool call è trasparente), in A2A l'agente chiamato è una **scatola opaca**: il chiamante vede solo l'avanzamento del Task e gli artefatti finali.

---
### 🧩 Concetti Chiave di A2A trattati:
1. **Agent Card (`/.well-known/agent-card.json`)**: il "biglietto da visita" con cui un agente pubblica le proprie skill, endpoint e schemi di I/O.
2. **Task Lifecycle**: la macchina a stati del task (`submitted` ➔ `working` ➔ `input-required` ➔ `completed` / `failed`).
3. **Messages & Parts**: scambio di messaggi formattati (`text`, `data`, `files`).
4. **Artifacts**: il risultato finale strutturato generato dall'agente destinatario.

---
### 🏢 Scenario del Caso Guida
1. L'**Agente Acquisti Interno (HR/Internal)** riceve una richiesta di acquisto autorizzata.
2. Contatta l'**Agente Forniture IT (IT Procurement Agent)** (gestito da un partner logistico esterno).
3. L'agente logistico analizza la richiesta, rileva che manca l'informazione sulla sede di consegna e transiziona il task nello stato formale **`input-required`**.
4. L'agente interno fornisce il dato mancante e il task si conclude in **`completed`** con l'emissione dell'**`Artifact`** di spedizione.

## 1. Discovery tramite Agent Card (`/.well-known/agent-card.json`)
Il primo passo del protocollo A2A è la *discovery*: l'agente chiamante interroga l'Agent Card dell'agente fornitore per scoprire quali skill espone e come interagirvi.

In [2]:
import json
from pprint import pprint
from a2a_simulation import ProcurementITAgent, BaseA2AAgent, TaskStatus

# Creiamo l'istanza dell'Agente IT Forniture (Fornitore esterno)
it_procurement_agent = ProcurementITAgent()

# Ispezioniamo la sua Agent Card
agent_card = it_procurement_agent.get_agent_card()
print("📋 AGENT CARD (/.well-known/agent-card.json):")
print(json.dumps(agent_card, indent=2, ensure_ascii=False))

📋 AGENT CARD (/.well-known/agent-card.json):
{
  "a2a_version": "1.0",
  "agent_id": "agent.it.procurement",
  "name": "IT Procurement & Logistics Agent",
  "description": "Agente specializzato nella verifica forniture hardware aziendali, giacenze e spedizioni.",
  "endpoint": "https://api.internal-corp.local/a2a/v1/procurement",
  "authentication": {
    "type": "bearer",
    "required": false
  },
  "skills": [
    {
      "id": "order_hardware_device",
      "name": "Ordinazione Hardware e Verifica Stock",
      "description": "Verifica la disponibilità di un device hardware e procede con il pre-ordine se autorizzato.",
      "input_schema": {
        "sku": "string",
        "quantity": "integer",
        "urgency": "string (optional)"
      },
      "output_schema": {
        "order_id": "string",
        "eta_days": "integer",
        "status": "string"
      }
    }
  ]
}


## 2. Creazione dell'Agente Chiamante (Internal Assistant)
Creiamo l'Agente Interno Aziendale che fungerà da orchestratore.

In [3]:
class InternalCompanyAgent(BaseA2AAgent):
    def __init__(self):
        super().__init__(
            agent_id="agent.internal.assistant",
            name="Internal Employee Assistant",
            description="Agente aziendale interno per supporto dipendenti e richieste acquisti.",
            endpoint="https://internal.corp/a2a/v1"
        )
        
    def process_task(self, task):
        # L'agente chiamante coordina le richieste
        pass

# Inizializziamo l'agente interno
internal_agent = InternalCompanyAgent()
print(f"✅ Agente interno inizializzato: {internal_agent.name} ({internal_agent.agent_id})")

✅ Agente interno inizializzato: Internal Employee Assistant (agent.internal.assistant)


## 3. Scambio di Messaggi A2A e Macchina a Stati del Task
Simuliamo lo scambio completo:
1. `internal_agent` invia un task con `skill: order_hardware_device` richiedendo il laptop `LAP-DEV-01`.
2. L'agente IT fornitore riceve il task e transiziona lo stato da `submitted` a `working`.
3. Mancando la sede aziendale di consegna, il fornitore transiziona lo stato in **`input-required`** e richiede l'informazione.
4. L'agente interno risponde con la sede (`Milano HQ`).
5. Il task passa a **`completed`** con l'inclusione di un **Artifact** contenente l'ID ordine e tempi di consegna.

In [4]:
print("STEP 1: Invio del Task dall'Agente Interno all'Agente IT Forniture")

initial_message_parts = [
    {
        "kind": "text",
        "text": "Richiesta emissione ordine per SKU: LAP-DEV-01 (Quantità: 1) per dipendente Mario Rossi (E4521)."
    }
]

# Invio del Task
task = internal_agent.send_task(
    target_agent=it_procurement_agent,
    skill_id="order_hardware_device",
    input_parts=initial_message_parts
)

print(f"Task ID generato: {task.task_id}")
print(f"Stato attuale del Task: 👉 [{task.status.value.upper()}]")
print(f"Ultimo messaggio ricevuto:")
pprint(task.history[-1])

STEP 1: Invio del Task dall'Agente Interno all'Agente IT Forniture
Task ID generato: task_8254ead7
Stato attuale del Task: 👉 [INPUT-REQUIRED]
Ultimo messaggio ricevuto:
{'parts': [{'kind': 'text',
            'text': 'Richiesta ricevuta per LAP-DEV-01. Specificare la sede di '
                    "destinazione (es. 'Milano HQ' o 'Roma Tech Hub') per "
                    'confermare la logistica.'}],
 'role': 'assistant'}


### Risoluzione dello stato `input-required`
Poiché il task si trova nello stato `input-required`, il protocollo prevede che il chiamante invii i dati integrativi tramite il metodo formale di fornitura input.

In [5]:
print("STEP 2: Fornitura dei dettagli richiesti (Sede di destinazione)")

additional_parts = [
    {
        "kind": "text",
        "text": "La sede di destinazione per la consegna è Milano HQ - Piano 3, Ufficio Tech."
    }
]

# Forniamo l'input per sbloccare il task
it_procurement_agent.provide_input(task.task_id, additional_parts)

print(f"Nuovo Stato del Task: 👉 [{task.status.value.upper()}]")
print(f"Ultimo messaggio dell'agente fornitore:")
pprint(task.history[-1])

STEP 2: Fornitura dei dettagli richiesti (Sede di destinazione)
Nuovo Stato del Task: 👉 [COMPLETED]
Ultimo messaggio dell'agente fornitore:
{'parts': [{'kind': 'text',
            'text': 'Ordine ORD-IT-55ABB7 approvato ed inserito nella coda di '
                    'spedizione. Consegna prevista in 3 giorni lavorativi.'}],
 'role': 'assistant'}


## 4. Ispezione degli Artifacts e della Traccia degli Stati
Al termine del task, l'agente fornitore ha prodotto un **Artifact** strutturato con gli estremi dell'ordine.

In [6]:
print("ARTIFACTS PRODOTTI:")
pprint(task.artifacts)

print("\n" + "="*60 + "\n")
print("STORICO TRANSIZIONI DI STATO DEL TASK:")
for t in task.state_transitions:
    print(f"  • {t['from']} ➔ {t['to']} | Nota: {t.get('note', '')}")

ARTIFACTS PRODOTTI:
[{'data': {'assigned_cost_eur': 1650.0,
           'delivery_hub': 'Milano HQ',
           'device': 'ThinkPad P16s Gen 2 (LAP-DEV-01)',
           'eta_days': 3,
           'order_id': 'ORD-IT-55ABB7',
           'status': 'APPROVED_AND_QUEUED'},
  'kind': 'artifact',
  'type': 'procurement_confirmation'}]


STORICO TRANSIZIONI DI STATO DEL TASK:
  • None ➔ submitted | Nota: 
  • submitted ➔ working | Nota: Task accettato dall'agente destinatario
  • working ➔ input-required | Nota: Manca l'indirizzo della sede aziendale di consegna per l'hardware selezionato.
  • input-required ➔ working | Nota: Nuovo input fornito dal chiamante
  • working ➔ completed | Nota: Disponibilità verificata e ordine registrato.


## 5. Integrazione con Ollama: Generazione del Messaggio A2A con LLM
Ora utilizziamo il modello **`llama3.2`** locale per generare automaticamente la richiesta A2A partendo dal linguaggio naturale del dipendente.

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3.2"

def generate_a2a_message_with_llm(user_prompt: str, target_agent_card: dict) -> dict:
    """
    Usa Llama 3.2 per strutturare una richiesta conforme alle skill dell'Agent Card.
    """
    prompt = f"""
Sei un assistente AI incaricato di generare una richiesta standard A2A (Agent2Agent) verso un agente fornitore.
Ecco l'Agent Card dell'agente destinatario:
{json.dumps(target_agent_card, indent=2, ensure_ascii=False)}

Richiesta dell'utente: "{user_prompt}"

Genera ESCLUSIVAMENTE un oggetto JSON valido con la seguente struttura:
{{
  "skill_id": "id_della_skill_scelta",
  "text_message": "testo formattato e dettagliato per la richiesta"
}}
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "format": "json"
    }
    
    try:
        resp = requests.post(OLLAMA_URL, json=payload)
        if resp.status_code == 200:
            content = resp.json()["message"]["content"]
            return json.loads(content)
        else:
            print("Errore Ollama:", resp.text)
            return None
    except Exception as e:
        print("Errore chiamata:", e)
        return None

# Test con Llama 3.2
richiesta_utente = "Ordina con urgenza il laptop LAP-DEV-01 per Mario Rossi"
a2a_payload = generate_a2a_message_with_llm(richiesta_utente, agent_card)
print("🤖 Payload generato da Llama 3.2 per A2A:")
pprint(a2a_payload)

🤖 Payload generato da Llama 3.2 per A2A:
{'skill_id': 'order_hardware_device',
 'text_message': '<b>Ordinazione Hardware e Verifica Stock</b>\n'
                 '\n'
                 'Per favore confermi se puoi verificare la disponibilità di '
                 'un device hardware e procedere con il pre-ordine per il '
                 'laptop LAP-DEV-01.\n'
                 'Il laptop LAP-DEV-01 è destinato a Mario Rossi.\n'
                 '\n'
                 "Dettagli dell'ordine:\n"
                 '\n'
                 '* Nome del device: LAP-DEV-01\n'
                 '* Quantità: 1\n'
                 '* Urgenza: Alta'}


: 

## 6. Confronto: MCP vs A2A
Riepiloghiamo la differenza concettuale fondamentale tra i due protocolli del Giorno 2:

| Caratteristica | **MCP (Model Context Protocol)** | **A2A (Agent2Agent)** |
| :--- | :--- | :--- |
| **Relazione** | Agente ➔ Tool / Dati | Agente ➔ Agente |
| **Trasparenza** | **Trasparente**: l'agente vede ogni singola query e tool | **Opaca**: l'agente vede solo il Task e gli Artifacts |
| **Ciclo di vita** | Chiamata di funzione sincrona / stream I/O | Task asincrono con stati (`working`, `input-required`) |
| **Scopo Primario** | Dare mani e occhi a un singolo modello | Far collaborare agenti autonomi di team o vendor diversi |

---
### 💡 Esercizio Pratico per lo Studente
**Obiettivo:** Creare un nuovo Agente A2A specializzato: `FinanceApprovalAgent` con la skill `approve_budget_expense`.

Implementa l'agente in modo che:
1. Se l'importo è `<= 1000€`, risponde direttamente con stato `completed` e un artifact con esito `APPROVED`.
2. Se l'importo è `> 1000€`, passa a `input-required` chiedendo `"Fornire il codice centro di costo (Cost Center) per approvazione manageriale"`.
3. Ricevuto il centro di costo, completa il task.

In [ ]:
# ESERCIZIO: Implementa FinanceApprovalAgent
class FinanceApprovalAgent(BaseA2AAgent):
    def __init__(self):
        super().__init__(
            agent_id="agent.finance.approval",
            name="Finance & Budget Approval Agent",
            description="Agente responsabile dell'autorizzazione spese e allocazione centri di costo.",
            endpoint="https://finance.corp/a2a/v1"
        )
        # TODO: Registra la skill 'approve_budget_expense'
        
    def process_task(self, task: A2ATask):
        # TODO: Implementa la logica dello stato
        pass

# Test del tuo agente Finance
print("Test del FinanceApprovalAgent personalizzato!")

---
### ✅ Recap di Lab 2
- Abbiamo compreso come funziona il **Protocollo A2A**.
- Abbiamo implementato la gestione degli stati di un **Task**, inclusa la sospensione per **`input-required`**.
- Abbiamo strutturato gli **Artifacts** di output.
- Nel prossimo notebook (Lab 3), applicherai questi principi al **tuo Project Canvas personale**!